# Synthetic Data Generation

Generates additional synthetic images for **scratch** and **deformation** classes.

**Run order:** G1 → G2 → G3 → G4 → G5 → G6



In [15]:
# ── CELL G1 — IMPORTS + CONFIG ──

import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import albumentations as A
from tqdm import tqdm
import shutil
import random

random.seed(42)
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────
CSV_PATH       = '/home/sufi/merged_dataset_metadata_augmented.csv'
SYNTHETIC_ROOT = '/home/sufi/lora_training/synthetic'
OUTPUT_CSV     = '/home/sufi/merged_dataset_metadata_augmented.csv'
BACKUP_CSV     = '/home/sufi/merged_dataset_metadata_augmented_backup.csv'

# ── Target counts ──────────────────────────────────────────────────────
TARGET_TOTAL   = 700
CLASSES_TO_GEN = ['fracture', 'minor_defect']

# ── Defect group map ───────────────────────────────────────────────────
DEFECT_GROUP_MAP = {
    # raw dataset labels → semantic group
    'crack':               'fracture',
    'fracture':            'fracture',
    'faulty_imprint':      'fracture',
    'hole':                'hole_void',
    'void':                'hole_void',
    'pit':                 'hole_void',
    'blowhole':            'hole_void',
    'scratch':             'scratch',
    'score':               'scratch',
    'stain':               'surface_quality',
    'color':               'surface_quality',
    'rough':               'surface_quality',
    'uneven':              'surface_quality',
    'inclusion':           'surface_quality',
    'discolor':            'surface_quality',
    'pilling':             'surface_quality',
    'bent':                'deformation',
    'bent_lead':           'deformation',
    'squeeze':             'deformation',
    'deformation':         'deformation',
    'contamination':       'contamination',
    'glue':                'contamination',
    'oil':                 'contamination',
    'glue_strip':          'contamination',
    'liquid':              'contamination',
    'metal_contamination': 'contamination',
    'missing':             'minor_defect',
    'misplaced':           'minor_defect',
    'flip':                'minor_defect',
    'missing_hole':        'minor_defect',
    'thread':              'minor_defect',
    'cable_swap':          'minor_defect',
    'combined':            'minor_defect',
    'cut':                 'cut',
    # ── direct semantic group name passthroughs ────────────────────
    'contamination_group': 'contamination',
    'hole_void':           'hole_void',
    'surface_quality':     'surface_quality',
    'minor_defect':        'minor_defect',
    'fracture_group':      'fracture',
}

SEMANTIC_GROUPS    = ['contamination','cut','deformation','fracture',
                      'hole_void','minor_defect','scratch','surface_quality']
DEFECT_TYPE_TO_IDX = {name: idx for idx, name in enumerate(SEMANTIC_GROUPS)}

def get_semantic(raw):
    if pd.isna(raw):
        return None
    raw_str = str(raw).strip().lower()
    # direct match first
    if raw_str in DEFECT_GROUP_MAP:
        return DEFECT_GROUP_MAP[raw_str]
    # semantic group name itself
    if raw_str in SEMANTIC_GROUPS:
        return raw_str
    # partial match fallback
    for k, v in DEFECT_GROUP_MAP.items():
        if k in raw_str:
            return v
    return None

print("✅ CELL G1 COMPLETE")
print(f"   Target per class : {TARGET_TOTAL}")
print(f"   Classes to gen   : {CLASSES_TO_GEN}")
print(f"   Output CSV       : {OUTPUT_CSV}")

✅ CELL G1 COMPLETE
   Target per class : 700
   Classes to gen   : ['fracture', 'minor_defect']
   Output CSV       : /home/sufi/merged_dataset_metadata_augmented.csv


In [1]:
# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC_GEN CELL G1 — Fix 1 (run in synthetic_gen.ipynb)
# ════════════════════════════════════════════════════════════════════════
 
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import albumentations as A
from tqdm import tqdm
import shutil
import random
 
random.seed(42)
np.random.seed(42)
 
CSV_PATH       = '/home/sufi/merged_dataset_metadata_augmented.csv'
SYNTHETIC_ROOT = '/home/sufi/lora_training/synthetic'
OUTPUT_CSV     = '/home/sufi/merged_dataset_metadata_augmented.csv'
BACKUP_CSV     = '/home/sufi/merged_dataset_metadata_augmented_backup2.csv'
 
# ── Target counts ──────────────────────────────────────────────────────
TARGET_TOTAL   = 600
CLASSES_TO_GEN = ['cut', 'hole_void', 'surface_quality']
 
# ── Defect group map ───────────────────────────────────────────────────
DEFECT_GROUP_MAP = {
    'crack':               'fracture',
    'fracture':            'fracture',
    'faulty_imprint':      'fracture',
    'hole':                'hole_void',
    'void':                'hole_void',
    'pit':                 'hole_void',
    'blowhole':            'hole_void',
    'scratch':             'scratch',
    'score':               'scratch',
    'stain':               'surface_quality',
    'color':               'surface_quality',
    'rough':               'surface_quality',
    'uneven':              'surface_quality',
    'inclusion':           'surface_quality',
    'discolor':            'surface_quality',
    'pilling':             'surface_quality',
    'bent':                'deformation',
    'bent_lead':           'deformation',
    'squeeze':             'deformation',
    'deformation':         'deformation',
    'contamination':       'contamination',
    'glue':                'contamination',
    'oil':                 'contamination',
    'glue_strip':          'contamination',
    'liquid':              'contamination',
    'metal_contamination': 'contamination',
    'missing':             'minor_defect',
    'misplaced':           'minor_defect',
    'flip':                'minor_defect',
    'missing_hole':        'minor_defect',
    'thread':              'minor_defect',
    'cable_swap':          'minor_defect',
    'combined':            'minor_defect',
    'cut':                 'cut',
    'hole_void':           'hole_void',
    'surface_quality':     'surface_quality',
    'minor_defect':        'minor_defect',
}
 
SEMANTIC_GROUPS    = ['contamination','cut','deformation','fracture',
                      'hole_void','minor_defect','scratch','surface_quality']
DEFECT_TYPE_TO_IDX = {name: idx for idx, name in enumerate(SEMANTIC_GROUPS)}
 
def get_semantic(raw):
    if pd.isna(raw):
        return None
    raw_str = str(raw).strip().lower()
    if raw_str in DEFECT_GROUP_MAP:
        return DEFECT_GROUP_MAP[raw_str]
    if raw_str in SEMANTIC_GROUPS:
        return raw_str
    for k, v in DEFECT_GROUP_MAP.items():
        if k in raw_str:
            return v
    return None
 
# ── Step A: Remove excess minor_defect synthetic rows from CSV ─────────
print("Step A: Cleaning excess minor_defect synthetic rows...")
df = pd.read_csv(CSV_PATH)
if 'path' in df.columns and 'image_path' not in df.columns:
    df = df.rename(columns={'path': 'image_path', 'category': 'product_type'})
 
shutil.copy(CSV_PATH, BACKUP_CSV)
print(f"   Backup saved → {BACKUP_CSV}")
 
df['defect_group_clean'] = df['defect_type'].apply(get_semantic)
 
# count minor_defect synthetic rows
md_syn_mask = (
    (df['defect_group_clean'] == 'minor_defect') &
    (df['image_path'].str.contains('synthetic', na=False))
)
n_md_syn    = md_syn_mask.sum()
md_real_mask = (
    (df['defect_group_clean'] == 'minor_defect') &
    (~df['image_path'].str.contains('synthetic', na=False))
)
n_md_real   = md_real_mask.sum()
TARGET_MD   = 250   # keep only 250 total minor_defect
keep_syn    = max(0, TARGET_MD - n_md_real)
 
print(f"   minor_defect real     : {n_md_real}")
print(f"   minor_defect synthetic: {n_md_syn}")
print(f"   Keeping synthetic     : {keep_syn}")
 
# keep only first `keep_syn` synthetic minor_defect rows
md_syn_rows    = df[md_syn_mask].index.tolist()
rows_to_drop   = md_syn_rows[keep_syn:]
df             = df.drop(index=rows_to_drop).reset_index(drop=True)
df_clean       = df.drop(columns=['defect_group_clean'])
df_clean.to_csv(CSV_PATH, index=False)
print(f"   CSV cleaned: {len(df):,} rows remaining")
print(f"   ✅ Step A complete\n")
 
print(f"✅ SYNTHETIC GEN CELL G1 COMPLETE")
print(f"   Target per class : {TARGET_TOTAL}")
print(f"   Classes to gen   : {CLASSES_TO_GEN}")
print(f"   Now run G2 → G3 → G4 → G5 → G6")

/home/sufi/miniconda3/envs/organized/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step A: Cleaning excess minor_defect synthetic rows...
   Backup saved → /home/sufi/merged_dataset_metadata_augmented_backup2.csv
   minor_defect real     : 198
   minor_defect synthetic: 1060
   Keeping synthetic     : 52
   CSV cleaned: 20,504 rows remaining
   ✅ Step A complete

✅ SYNTHETIC GEN CELL G1 COMPLETE
   Target per class : 600
   Classes to gen   : ['cut', 'hole_void', 'surface_quality']
   Now run G2 → G3 → G4 → G5 → G6


In [2]:
# ── CELL G2 — INSPECT EXISTING DATA ──

df = pd.read_csv(CSV_PATH)

# ── Rename columns if needed ───────────────────────────────────────────
if 'path' in df.columns and 'image_path' not in df.columns:
    df = df.rename(columns={'path': 'image_path'})
if 'category' in df.columns and 'product_type' not in df.columns:
    df = df.rename(columns={'category': 'product_type'})

print(f"CSV loaded: {len(df):,} rows")
print(f"Columns   : {list(df.columns)}\n")

# ── Backup CSV before modifying ────────────────────────────────────────
shutil.copy(CSV_PATH, BACKUP_CSV)
print(f"✅ Backup saved → {BACKUP_CSV}\n")

# ── Apply defect group map to get semantic labels ──────────────────────
def get_semantic(raw):
    if pd.isna(raw):
        return None
    return DEFECT_GROUP_MAP.get(str(raw).strip().lower(), None)

df['defect_group_clean'] = df['defect_type'].apply(get_semantic)

# ── Count per class ────────────────────────────────────────────────────
print(f"{'Class':<20} {'Real':>8} {'Synthetic':>12} {'Total':>8} {'Need':>8}")
print("─" * 58)

class_stats = {}
for cls in CLASSES_TO_GEN:
    real_mask = (
        (df['defect_group_clean'] == cls) &
        (~df['image_path'].str.contains('synthetic', na=False))
    )
    syn_mask = (
        (df['defect_group_clean'] == cls) &
        (df['image_path'].str.contains('synthetic', na=False))
    )
    n_real = real_mask.sum()
    n_syn  = syn_mask.sum()
    n_total = n_real + n_syn
    n_need  = max(0, TARGET_TOTAL - n_total)

    class_stats[cls] = {
        'real_df'    : df[real_mask].copy(),
        'n_real'     : n_real,
        'n_syn'      : n_syn,
        'n_total'    : n_total,
        'n_need'     : n_need,
        'real_paths' : df[real_mask]['image_path'].tolist(),
    }
    print(f"  {cls:<18} {n_real:>8} {n_syn:>12} {n_total:>8} {n_need:>8}")

print("─" * 58)
print("\n✅ CELL G2 COMPLETE")

CSV loaded: 20,504 rows
Columns   : ['image_path', 'dataset', 'product_type', 'defect_type', 'label', 'split', 'defect_group', 'defect_type_label']

✅ Backup saved → /home/sufi/merged_dataset_metadata_augmented_backup2.csv

Class                    Real    Synthetic    Total     Need
──────────────────────────────────────────────────────────
  cut                      53          385      438      162
  hole_void               275          269      544       56
  surface_quality         331          307      638        0
──────────────────────────────────────────────────────────

✅ CELL G2 COMPLETE


In [3]:
# ── CELL G3 — AUGMENTATION PIPELINES ──

# ── Fracture augmentation ──────────────────────────────────────────────
# Fracture = cracks, thin breaks → directional + high contrast
fracture_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.15,
        rotate_limit=45, p=0.7
    ),
    A.OneOf([
        A.ElasticTransform(alpha=80, sigma=5, p=1.0),
        A.GridDistortion(num_steps=4, distort_limit=0.2, p=1.0),
        A.OpticalDistortion(distort_limit=0.2, p=1.0),
    ], p=0.5),
    A.OneOf([
        A.CLAHE(clip_limit=6.0, p=1.0),
        A.RandomBrightnessContrast(
            brightness_limit=0.3, contrast_limit=0.4, p=1.0),
        A.Sharpen(alpha=(0.3, 0.7), lightness=(0.5, 1.0), p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 40), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4), p=1.0),
    ], p=0.4),
    A.OneOf([
        A.ToGray(p=1.0),
        A.HueSaturationValue(
            hue_shift_limit=5, sat_shift_limit=20,
            val_shift_limit=20, p=1.0),
    ], p=0.3),
])

# ── Minor defect augmentation ──────────────────────────────────────────
# Minor defect = small missing/misplaced parts → subtle transforms
minor_defect_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.2,
        rotate_limit=20, p=0.7
    ),
    A.OneOf([
        A.ElasticTransform(alpha=60, sigma=4, p=1.0),
        A.GridDistortion(num_steps=4, distort_limit=0.2, p=1.0),
        A.Perspective(scale=(0.03, 0.10), p=1.0),
    ], p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(
            brightness_limit=0.25, contrast_limit=0.25, p=1.0),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=25,
            val_shift_limit=20, p=1.0),
        A.CLAHE(clip_limit=3.0, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(5, 30), p=1.0),
        A.MultiplicativeNoise(multiplier=(0.92, 1.08), p=1.0),
    ], p=0.4),
    A.CoarseDropout(
        max_holes=3, max_height=15, max_width=15,
        min_holes=1, fill_value=0, p=0.3
    ),
])

AUG_MAP = {
    'fracture'    : fracture_aug,
    'minor_defect': minor_defect_aug,
}

print("✅ CELL G3 COMPLETE")
print("   Fracture augmentation     : directional + high contrast")
print("   Minor defect augmentation : subtle geometric + noise")

✅ CELL G3 COMPLETE
   Fracture augmentation     : directional + high contrast
   Minor defect augmentation : subtle geometric + noise


/home/sufi/miniconda3/envs/organized/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_34030/2029649350.py:25: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 40), p=1.0),
/tmp/ipykernel_34030/2029649350.py:60: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 30), p=1.0),
/tmp/ipykernel_34030/2029649350.py:63: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


In [5]:
# ── CELL G3 — AUGMENTATION PIPELINES ──

# ── Cut augmentation ───────────────────────────────────────────────────
cut_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                       rotate_limit=45, p=0.7),
    A.OneOf([
        A.ElasticTransform(alpha=80, sigma=5, p=1.0),
        A.GridDistortion(num_steps=4, distort_limit=0.2, p=1.0),
        A.OpticalDistortion(distort_limit=0.2, p=1.0),
    ], p=0.5),
    A.OneOf([
        A.CLAHE(clip_limit=5.0, p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.3,
                                   contrast_limit=0.4, p=1.0),
        A.Sharpen(alpha=(0.3, 0.7), lightness=(0.5, 1.0), p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 40), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.05),
                   intensity=(0.1, 0.4), p=1.0),
    ], p=0.4),
])

# ── Hole/void augmentation ─────────────────────────────────────────────
hole_void_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2,
                       rotate_limit=30, p=0.7),
    A.OneOf([
        A.ElasticTransform(alpha=60, sigma=4, p=1.0),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.OpticalDistortion(distort_limit=0.3, p=1.0),
    ], p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3,
                                   contrast_limit=0.3, p=1.0),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=25,
                             val_shift_limit=20, p=1.0),
        A.CLAHE(clip_limit=4.0, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50), p=1.0),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
    ], p=0.4),
    A.CoarseDropout(max_holes=3, max_height=25, max_width=25,
                    min_holes=1, fill_value=0, p=0.3),
])

# ── Surface quality augmentation ───────────────────────────────────────
surface_quality_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                       rotate_limit=20, p=0.6),
    A.OneOf([
        A.ElasticTransform(alpha=50, sigma=4, p=1.0),
        A.GridDistortion(num_steps=4, distort_limit=0.2, p=1.0),
    ], p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.35,
                                   contrast_limit=0.35, p=1.0),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=35,
                             val_shift_limit=25, p=1.0),
        A.CLAHE(clip_limit=5.0, p=1.0),
        A.ToGray(p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50), p=1.0),
        A.ISONoise(p=1.0),
        A.MultiplicativeNoise(multiplier=(0.88, 1.12), p=1.0),
    ], p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.Sharpen(alpha=(0.2, 0.5), p=1.0),
        A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.7), p=1.0),
    ], p=0.4),
])

AUG_MAP = {
    'cut'            : cut_aug,
    'hole_void'      : hole_void_aug,
    'surface_quality': surface_quality_aug,
}

print("✅ CELL G3 COMPLETE")
print("   cut             : directional + contrast")
print("   hole_void       : geometric + dropout")
print("   surface_quality : color + texture heavy")

✅ CELL G3 COMPLETE
   cut             : directional + contrast
   hole_void       : geometric + dropout
   surface_quality : color + texture heavy


/tmp/ipykernel_34030/4119632923.py:22: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 40), p=1.0),
/tmp/ipykernel_34030/4119632923.py:48: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=1.0),
/tmp/ipykernel_34030/4119632923.py:51: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=3, max_height=25, max_width=25,
/tmp/ipykernel_34030/4119632923.py:75: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=1.0),


In [6]:
# ── CELL G4 — GENERATE SYNTHETIC IMAGES ──

IMG_SIZE    = 224
new_rows    = []
gen_summary = {}

for cls in CLASSES_TO_GEN:
    stats     = class_stats[cls]
    n_need    = stats['n_need']
    real_paths = [p for p in stats['real_paths'] if os.path.exists(p)]

    if n_need <= 0:
        print(f"⏭  {cls}: already has {stats['n_total']} ≥ {TARGET_TOTAL}, skipping")
        gen_summary[cls] = 0
        continue

    if len(real_paths) == 0:
        print(f"❌ {cls}: no real images found — check paths in CSV")
        gen_summary[cls] = 0
        continue

    print(f"\n🔧 Generating {n_need} synthetic images for '{cls}'")
    print(f"   Source pool: {len(real_paths)} real images")

    # ── output directory ──────────────────────────────────────────
    out_dir = Path(SYNTHETIC_ROOT) / cls
    out_dir.mkdir(parents=True, exist_ok=True)

    # ── count existing synthetic files to avoid overwriting ───────
    existing = list(out_dir.glob('*.jpg')) + list(out_dir.glob('*.png'))
    start_idx = len(existing)
    print(f"   Existing synthetic: {start_idx}")

    aug_fn     = AUG_MAP[cls]
    generated  = 0
    failed     = 0

    # ── get product_type and dataset from real rows ────────────────
    real_df   = stats['real_df'].reset_index(drop=True)
    split_map = real_df.set_index('image_path')['split'].to_dict()

    pbar = tqdm(total=n_need, desc=f"  {cls}", unit="img")

    while generated < n_need:
        # pick a random real image
        src_path = random.choice(real_paths)
        src_row  = real_df[real_df['image_path'] == src_path]
        if len(src_row) == 0:
            src_row = real_df.sample(1)
        src_row  = src_row.iloc[0]

        # read image
        img = cv2.imread(src_path)
        if img is None:
            failed += 1
            if failed > n_need * 2:
                print(f"\n   ⚠️  Too many read failures, check paths")
                break
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # augment
        try:
            aug_img = aug_fn(image=img)['image']
        except Exception as e:
            failed += 1
            continue

        # save
        fname    = f"{cls}_syn_{start_idx + generated:05d}.jpg"
        out_path = str(out_dir / fname)
        save_img = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
        ok       = cv2.imwrite(out_path, save_img,
                               [cv2.IMWRITE_JPEG_QUALITY, 95])
        if not ok:
            failed += 1
            continue

        # build new CSV row — copy from source row
        new_row = src_row.to_dict()
        new_row['image_path']        = out_path
        new_row['defect_type']       = cls
        new_row['defect_group_clean']= cls
        new_row['defect_type_label'] = DEFECT_TYPE_TO_IDX[cls]
        new_row['label']             = 'defect'
        new_row['binary_label']      = 1
        new_row['split']             = 'train'  # all synthetic → train only
        new_rows.append(new_row)

        generated += 1
        pbar.update(1)

    pbar.close()
    gen_summary[cls] = generated
    print(f"   ✅ Generated: {generated}   Failed: {failed}")

print(f"\n{'═'*50}")
print(f"  GENERATION COMPLETE")
print(f"{'═'*50}")
for cls, cnt in gen_summary.items():
    print(f"  {cls:<18} +{cnt} new images")
print(f"  Total new rows  : {len(new_rows)}")
print(f"{'═'*50}")
print("\n✅ CELL G4 COMPLETE")


🔧 Generating 162 synthetic images for 'cut'
   Source pool: 53 real images
   Existing synthetic: 385


  cut: 100%|██████████| 162/162 [00:04<00:00, 37.19img/s]


   ✅ Generated: 162   Failed: 0

🔧 Generating 56 synthetic images for 'hole_void'
   Source pool: 275 real images
   Existing synthetic: 269


  hole_void: 100%|██████████| 56/56 [00:00<00:00, 198.30img/s]

   ✅ Generated: 56   Failed: 0
⏭  surface_quality: already has 638 ≥ 600, skipping

══════════════════════════════════════════════════
  GENERATION COMPLETE
══════════════════════════════════════════════════
  cut                +162 new images
  hole_void          +56 new images
  surface_quality    +0 new images
  Total new rows  : 218
══════════════════════════════════════════════════

✅ CELL G4 COMPLETE


In [7]:
# ── CELL G5 — UPDATE CSV ──

if len(new_rows) == 0:
    print("⚠️  No new rows to add — CSV unchanged")
else:
    new_df = pd.DataFrame(new_rows)

    # ── drop helper column ─────────────────────────────────────────
    if 'defect_group_clean' in df.columns:
        df = df.drop(columns=['defect_group_clean'])
    if 'defect_group_clean' in new_df.columns:
        new_df = new_df.drop(columns=['defect_group_clean'])

    # ── align columns ──────────────────────────────────────────────
    for col in df.columns:
        if col not in new_df.columns:
            new_df[col] = None
    new_df = new_df[df.columns]

    # ── concat + save ──────────────────────────────────────────────
    df_updated = pd.concat([df, new_df], ignore_index=True)
    df_updated.to_csv(OUTPUT_CSV, index=False)

    print(f"✅ CSV updated: {len(df):,} → {len(df_updated):,} rows")
    print(f"   Saved to: {OUTPUT_CSV}")
    print(f"   Backup at: {BACKUP_CSV}")

print("\n✅ CELL G5 COMPLETE")

✅ CSV updated: 20,504 → 20,722 rows
   Saved to: /home/sufi/merged_dataset_metadata_augmented.csv
   Backup at: /home/sufi/merged_dataset_metadata_augmented_backup2.csv

✅ CELL G5 COMPLETE


In [8]:
# ── CELL G6 — VERIFY + SUMMARY ──

df_verify = pd.read_csv(OUTPUT_CSV)
if 'path' in df_verify.columns and 'image_path' not in df_verify.columns:
    df_verify = df_verify.rename(columns={'path': 'image_path'})

df_verify['defect_group_clean'] = df_verify['defect_type'].apply(get_semantic)

print("="*60)
print("  📊 FINAL DATASET SUMMARY")
print("="*60)

all_classes = SEMANTIC_GROUPS
print(f"\n{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("─"*52)

for cls in all_classes:
    mask  = df_verify['defect_group_clean'] == cls
    train = ((df_verify['split'] == 'train') & mask).sum()
    val   = ((df_verify['split'] == 'val')   & mask).sum()
    test  = ((df_verify['split'] == 'test')  & mask).sum()
    total = mask.sum()
    flag  = ' ✅' if cls in CLASSES_TO_GEN else ''
    print(f"  {cls:<18} {train:>8} {val:>8} {test:>8} {total:>8}{flag}")

print("─"*52)
total_defect = (df_verify['defect_group_clean'].notna()).sum()
print(f"  {'TOTAL DEFECT':<18} {total_defect:>35}")
print(f"  {'TOTAL ROWS':<18} {len(df_verify):>35}")

# ── Verify images actually exist ──────────────────────────────────────
print(f"\n🔍 Spot-checking new synthetic files...")
for cls in CLASSES_TO_GEN:
    syn_dir = Path(SYNTHETIC_ROOT) / cls
    files   = list(syn_dir.glob('*.jpg')) + list(syn_dir.glob('*.png'))
    print(f"   {cls:<18} {len(files)} files in {syn_dir}")

    # check a sample is readable
    if files:
        sample = cv2.imread(str(random.choice(files)))
        if sample is not None:
            print(f"   {'':18} ✅ Images readable (shape={sample.shape})")
        else:
            print(f"   {'':18} ❌ WARNING: sample image unreadable!")

print(f"\n{'='*60}")
print(f"  ✅ ALL DONE — ready to retrain EdgeNet")
print(f"  Next steps:")
print(f"    1. Open your training notebook")
print(f"    2. Run Cell 77 → AUG FIX → Cell 78B → Cell 2 → Cell 3 → Cell 4")
print(f"    3. The updated CSV will be loaded automatically")
print(f"{'='*60}")

print("\n✅ CELL G6 COMPLETE")

  📊 FINAL DATASET SUMMARY

Class                   Train      Val     Test    Total
────────────────────────────────────────────────────
  contamination           899       33       12      944
  cut                     582       12        6      600 ✅
  deformation             784        9        7      800
  fracture                732       43       25      800
  hole_void               520       54       26      600 ✅
  minor_defect            147       26        9      182
  scratch                 771       18       11      800
  surface_quality         539       63       36      638 ✅
────────────────────────────────────────────────────
  TOTAL DEFECT                                      5364
  TOTAL ROWS                                       20722

🔍 Spot-checking new synthetic files...
   cut                547 files in /home/sufi/lora_training/synthetic/cut
                      ✅ Images readable (shape=(224, 224, 3))
   hole_void          325 files in /home/sufi/lora_trainin

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  IMPROVEMENT CELLS — Option 1 + 2 + 3 + TTA                        ║
# ║                                                                      ║
# ║  FULL RUN ORDER:                                                     ║
# ║  [NEW] Cell SYN1 → SYN2 → SYN3 → SYN4  (synthetic generation)      ║
# ║  Cell 77 → Cell 1 → AUG FIX → Cell 78                              ║
# ║  [NEW] Cell 78C  (CutMix dataloader)                                ║
# ║  [NEW] Cell 2B   (Loss with Label Smoothing)                        ║
# ║  Cell 3 → Cell 4                                                     ║
# ║  [NEW] Cell TTA  (after training, at inference)                      ║
# ╚══════════════════════════════════════════════════════════════════════╝

In [5]:
# ── CELL SYN1 FIX — run this instead of original SYN1 ─────────────────

import os, random
import cv2
import numpy as np
import pandas as pd
import albumentations as A
from pathlib import Path
from tqdm import tqdm

CSV_PATH       = '/home/sufi/merged_dataset_metadata_augmented.csv'
SYNTHETIC_ROOT = '/home/sufi/lora_training/synthetic'
IMG_SIZE       = 224

TARGETS = {
    'deformation': 1200,
    'scratch'    : 1100,
}

SEMANTIC_GROUPS    = ['contamination','cut','deformation','fracture',
                      'hole_void','minor_defect','scratch','surface_quality']
DEFECT_TYPE_TO_IDX = {n: i for i, n in enumerate(SEMANTIC_GROUPS)}

df = pd.read_csv(CSV_PATH)
if 'path' in df.columns and 'image_path' not in df.columns:
    df = df.rename(columns={'path':'image_path','category':'product_type'})

def compute_binary(v):
    if pd.isna(v): return 0
    return 0 if str(v).strip().lower() in (
        '0','good','normal','ok','casting_ok') else 1
df['binary_label'] = df['label'].apply(compute_binary)

# ── Remap to get defect_type_label ────────────────────────────────────
DEFECT_GROUP_MAP = {
    'crack':'fracture','fracture':'fracture','faulty_imprint':'fracture',
    'hole':'hole_void','void':'hole_void','pit':'hole_void','blowhole':'hole_void',
    'scratch':'scratch','score':'scratch',
    'stain':'surface_quality','color':'surface_quality','rough':'surface_quality',
    'uneven':'surface_quality','inclusion':'surface_quality',
    'discolor':'surface_quality','pilling':'surface_quality',
    'bent':'deformation','bent_lead':'deformation',
    'squeeze':'deformation','deformation':'deformation',
    'contamination':'contamination','glue':'contamination',
    'oil':'contamination','glue_strip':'contamination',
    'liquid':'contamination','metal_contamination':'contamination',
    'missing':'minor_defect','misplaced':'minor_defect','flip':'minor_defect',
    'missing_hole':'minor_defect','thread':'minor_defect',
    'cable_swap':'minor_defect','combined':'minor_defect',
    'cut':'cut',
    'hole_void':'hole_void','surface_quality':'surface_quality',
    'minor_defect':'minor_defect',
}

def remap_defect(raw):
    if pd.isna(raw): return -1
    s   = str(raw).strip().lower()
    sem = DEFECT_GROUP_MAP.get(s, None)
    if sem is None and s in SEMANTIC_GROUPS: sem = s
    if sem is None:
        for k, v in DEFECT_GROUP_MAP.items():
            if k in s: sem = v; break
    return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem, -1)

df['defect_type_label'] = df['defect_type'].apply(remap_defect)
df.loc[df['binary_label'] == 0, 'defect_type_label'] = -1

print("="*60)
print("  SYNTHETIC GENERATION — deformation + scratch")
print("="*60)

for cls, target in TARGETS.items():
    cls_idx   = DEFECT_TYPE_TO_IDX[cls]

    # ── Use defect_type_label for matching ─────────────────────────────
    real_mask = (
        (df['defect_type_label'] == cls_idx) &
        (~df['image_path'].str.contains('synthetic', na=False))
    )
    syn_mask  = (
        (df['defect_type_label'] == cls_idx) &
        (df['image_path'].str.contains('synthetic', na=False))
    )
    real_df   = df[real_mask].copy().reset_index(drop=True)
    n_real    = len(real_df)
    n_syn     = syn_mask.sum()
    n_total   = n_real + n_syn
    n_need    = max(0, target - n_total)

    print(f"\n  {cls} (label={cls_idx})")
    print(f"    Real sources: {n_real}  "
          f"Existing syn: {n_syn}  Total: {n_total}  Need: {n_need}")

    # Show raw defect_type values that map to this class
    raw_vals = df[real_mask]['defect_type'].value_counts().to_dict()
    print(f"    Raw values  : {raw_vals}")

    if n_real == 0:
        print(f"    ⚠️  No real images — will use synthetic as source pool")

print("\n✅ CELL SYN1 FIX COMPLETE")

  SYNTHETIC GENERATION — deformation + scratch

  deformation (label=2)
    Real sources: 96  Existing syn: 733  Total: 829  Need: 371
    Raw values  : {'bent': 37, 'squeeze': 20, 'squeezed_teeth': 16, 'bent_wire': 13, 'bent_lead': 10}

  scratch (label=6)
    Real sources: 140  Existing syn: 709  Total: 849  Need: 251
    Raw values  : {'scratch': 91, 'scratch_neck': 25, 'scratch_head': 24}

✅ CELL SYN1 FIX COMPLETE


In [11]:
# ── NUMPY COMPAT FIX — run once before any albumentations usage ───────
import numpy as np
if not hasattr(np, 'matrix'):
    np.matrix = np.ndarray   # shim for albumentations/scipy on numpy 2.x

# Also pin the albumentations version if the shim isn't enough:
# !pip install "albumentations==1.3.1" --quiet
print("✅ numpy compat shim applied")

✅ numpy compat shim applied


In [9]:
# ════════════════════════════════════════════════════════════════════════
# CELL SYN2 — AUGMENTATION PIPELINES (deformation + scratch specific)
# ════════════════════════════════════════════════════════════════════════
 
# Deformation: shape distortion, bending, warping
deformation_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.4),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.7),
    # Core: elastic/grid distortion mimics bending/deformation
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=6, p=1.0),
        A.GridDistortion(num_steps=6, distort_limit=0.4, p=1.0),
        A.OpticalDistortion(distort_limit=0.4, p=1.0),
    ], p=0.8),
    A.OneOf([
        A.RandomBrightnessContrast(
            brightness_limit=0.3, contrast_limit=0.4, p=1.0),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=30,
            val_shift_limit=25, p=1.0),
        A.CLAHE(clip_limit=5.0, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(p=1.0),
        A.MultiplicativeNoise(multiplier=(0.88, 1.12), p=1.0),
    ], p=0.4),
])
 
# Scratch: directional marks — NO elastic (makes it look like fracture)
scratch_aug = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    # Full rotation — scratches can be at any angle
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1, rotate_limit=180, p=0.8),
    # Surface texture enhancement — makes shallow marks visible
    A.OneOf([
        A.Sharpen(alpha=(0.5, 1.0), lightness=(0.5, 1.0), p=1.0),
        A.Emboss(alpha=(0.5, 1.0), strength=(0.5, 1.0), p=1.0),
        A.CLAHE(clip_limit=8.0, tile_grid_size=(4, 4), p=1.0),
    ], p=0.9),
    # Metallic sheen variation
    A.OneOf([
        A.HueSaturationValue(
            hue_shift_limit=5, sat_shift_limit=40,
            val_shift_limit=30, p=1.0),
        A.RandomBrightnessContrast(
            brightness_limit=0.2, contrast_limit=0.5, p=1.0),
        A.ToGray(p=1.0),
    ], p=0.7),
    A.GaussNoise(p=0.3),
    # NO ElasticTransform — would make scratch look like fracture
])
 
AUG_MAP = {
    'deformation': deformation_aug,
    'scratch'    : scratch_aug,
}
 
print("✅ CELL SYN2 COMPLETE")
print("   deformation : elastic/grid distortion (shape bending)")
print("   scratch     : directional sharpen/emboss (NO elastic)")
 
 


✅ CELL SYN2 COMPLETE
   deformation : elastic/grid distortion (shape bending)
   scratch     : directional sharpen/emboss (NO elastic)


In [ ]:
# ── CELL SYN3 FIX — key change: use defect_type_label not defect_type ─

new_rows    = []
gen_summary = {}

for cls, target in TARGETS.items():
    cls_idx   = DEFECT_TYPE_TO_IDX[cls]

    real_mask  = (
        (df['defect_type_label'] == cls_idx) &
        (~df['image_path'].str.contains('synthetic', na=False))
    )
    syn_mask   = (
        (df['defect_type_label'] == cls_idx) &
        (df['image_path'].str.contains('synthetic', na=False))
    )
    real_df    = df[real_mask].copy().reset_index(drop=True)
    real_paths = [p for p in real_df['image_path'].tolist()
                  if os.path.exists(p)]

    n_syn      = syn_mask.sum()
    n_real     = len(real_df)
    n_total    = n_real + n_syn
    n_need     = max(0, target - n_total)

    if n_need == 0:
        print(f"⏭  {cls}: already {n_total} ≥ {target}, skipping")
        gen_summary[cls] = 0
        


🔧 Generating 371 synthetic for 'deformation'
   Source pool : 96 images
   Existing    : 733 files in output dir


  deformation:   0%|          | 0/371 [00:00<?, ?img/s]

  deformation: 100%|██████████| 371/371 [00:06<00:00, 60.58img/s]


   ✅ Generated: 371   Failed: 0

🔧 Generating 251 synthetic for 'scratch'
   Source pool : 140 images
   Existing    : 1009 files in output dir


  scratch: 100%|██████████| 251/251 [00:04<00:00, 62.31img/s]

   ✅ Generated: 251   Failed: 0

═══════════════════════════════════════════════════════
  deformation          +371 new images
  scratch              +251 new images
  Total new rows : 622
═══════════════════════════════════════════════════════

✅ CELL SYN3 FIX COMPLETE — Run SYN4 to save CSV


In [ ]:
continue

    # ── If no real images, use existing synthetic as source pool ──────
    if len(real_paths) == 0:
        print(f"⚠️  {cls}: no real images — using synthetic as source")
        syn_paths = df[syn_mask]['image_path'].tolist()
        real_paths = [p for p in syn_paths if os.path.exists(p)]
        real_df    = df[syn_mask].copy().reset_index(drop=True)
        if len(real_paths) == 0:
            print(f"❌ {cls}: no source images at all — skipping")
            gen_summary[cls] = 0
            continue

    print(f"\n🔧 Generating {n_need} synthetic for '{cls}'")
    print(f"   Source pool : {len(real_paths)} images")

    out_dir = Path(SYNTHETIC_ROOT) / cls
    out_dir.mkdir(parents=True, exist_ok=True)

    existing  = list(out_dir.glob('*.jpg')) + list(out_dir.glob('*.png'))
    start_idx = len(existing)
    print(f"   Existing    : {start_idx} files in output dir")

    aug_fn    = AUG_MAP[cls]
    generated = 0
    failed    = 0

    pbar = tqdm(total=n_need, desc=f"  {cls}", unit="img")

    while generated < n_need:
        src_path = random.choice(real_paths)
        src_rows = real_df[real_df['image_path'] == src_path]
        src_row  = src_rows.iloc[0] if len(src_rows) > 0 \
                   else real_df.sample(1).iloc[0]


In [ ]:

        img = cv2.imread(src_path)
        if img is None:
            failed += 1
            if failed > n_need * 3: break
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        try:
            aug_img = aug_fn(image=img)['image']
        except Exception:
            failed += 1
            continue

        fname    = f"{cls}_syn_{start_idx + generated:05d}.jpg"
        out_path = str(out_dir / fname)
        save_img = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
        ok       = cv2.imwrite(out_path, save_img,
                               [cv2.IMWRITE_JPEG_QUALITY, 95])
        if not ok:
            failed += 1
            continue

        new_row = src_row.to_dict()
        new_row['image_path']         = out_path
        new_row['defect_type']        = cls
        new_row['defect_type_label']  = cls_idx
        new_row['label']              = 'defect'
        new_row['binary_label']       = 1
        new_row['split']              = 'train'
        new_rows.append(new_row)

        generated += 1
        pbar.update(1)

    pbar.close()
    gen_summary[cls] = generated
    print(f"   ✅ Generated: {generated}   Failed: {failed}")

print(f"\n{'═'*55}")
for cls, cnt in gen_summary.items():
    print(f"  {cls:<20} +{cnt} new images")
print(f"  Total new rows : {len(new_rows)}")
print(f"{'═'*55}")
print("\n✅ CELL SYN3 FIX COMPLETE — Run SYN4 to save CSV")

In [10]:
# ════════════════════════════════════════════════════════════════════════
# CELL SYN4 — SAVE UPDATED CSV + VERIFY
# ════════════════════════════════════════════════════════════════════════
 
if len(new_rows) > 0:
    backup_path = CSV_PATH.replace('.csv', '_backup_syn2.csv')
    df.to_csv(backup_path, index=False)
    print(f"✅ Backup saved → {backup_path}")
 
    new_df  = pd.DataFrame(new_rows)
    df_upd  = pd.concat([df, new_df], ignore_index=True)
    df_upd.to_csv(CSV_PATH, index=False)
    print(f"✅ CSV updated: {len(df):,} → {len(df_upd):,} rows")
    print(f"   Saved to: {CSV_PATH}")
 
    # Verify final class counts
    print(f"\n📊 Final class counts (deformation + scratch):")
    for cls in ['deformation', 'scratch']:
        mask_total = df_upd['defect_type'] == cls
        mask_real  = mask_total & (~df_upd['image_path'].str.contains(
            'synthetic', na=False))
        mask_syn   = mask_total & (df_upd['image_path'].str.contains(
            'synthetic', na=False))
        print(f"   {cls:<20} real={mask_real.sum():<6} "
              f"syn={mask_syn.sum():<6} total={mask_total.sum()}")
else:
    print("⚠️  No new rows generated — CSV not changed")
 
print("\n✅ CELL SYN4 COMPLETE")
print("   Now run: Cell 77 → Cell 1 → AUG FIX → Cell 78 → Cell 78C → Cell 2B → Cell 3 → Cell 4")


✅ Backup saved → /home/sufi/merged_dataset_metadata_augmented_backup_syn2.csv
✅ CSV updated: 20,722 → 21,344 rows
   Saved to: /home/sufi/merged_dataset_metadata_augmented.csv

📊 Final class counts (deformation + scratch):
   deformation          real=0      syn=1104   total=1104
   scratch              real=91     syn=960    total=1051

✅ CELL SYN4 COMPLETE
   Now run: Cell 77 → Cell 1 → AUG FIX → Cell 78 → Cell 78C → Cell 2B → Cell 3 → Cell 4
